In [1]:
# 1. Install necessary libraries
!pip install -q langgraph langchain-groq langchain-community langchain-core requests ddgs

import os
import re
import requests
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver  # Needed for conversation memory

# 2. Setup API Key and LLM
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

# 3. Define Tools
search_tool = DuckDuckGoSearchRun()

@tool
def get_weather_data(city: str) -> str:
    """
    Fetches the current weather data for a given city.
    Use this when user asks about weather, temperature, humidity or wind speed.
    """
    geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1"
    geo = requests.get(geo_url).json()

    if not geo.get("results"):
        return f"City '{city}' not found."

    lat = geo["results"][0]["latitude"]
    lon = geo["results"][0]["longitude"]
    name = geo["results"][0]["name"]
    country = geo["results"][0].get("country", "")

    weather_url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={lat}&longitude={lon}"
        f"&current=temperature_2m,relative_humidity_2m,"
        f"wind_speed_10m,precipitation,weather_code"
        f"&timezone=auto"
    )
    weather = requests.get(weather_url).json()
    current = weather["current"]

    return {
        "city": f"{name}, {country}",
        "temperature_C": current["temperature_2m"],
        "humidity_%": current["relative_humidity_2m"],
        "wind_speed_kmh": current["wind_speed_10m"],
        "precipitation_mm": current["precipitation"],
    }


# 4. Initialize Memory and Agent
memory = MemorySaver()

# Define the persona/instructions
system_message = "You are a witty, helpful AI assistant named Gorq. Use a friendly tone and always summarize search results concisely."

# Use 'prompt' instead of 'state_modifier' to fix the TypeError
agent_executor = create_react_agent(
    model=llm,
    tools=[search_tool, get_weather_data],
    checkpointer=memory,
    prompt=system_message  # Changed this line
)

print("✅ Agent ready! Personality 'Gorq' is now active.")

# 5. Continuous Chat Loop
# thread_id acts as a unique ID for this specific conversation
config = {"configurable": {"thread_id": "chat_session_1"}}

print("Start texting (type 'exit' or 'quit' to stop)")
print("-" * 50)

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit", "bye"]:
        print("🤖: Goodbye!")
        break

    try:
        # We pass the config so the agent knows which history to load
        response = agent_executor.invoke(
            {"messages": [("user", user_input)]},
            config=config
        )

        # Extract the last message from the agent
        output = response["messages"][-1].content

        # Format and display the response
        print("\n" + "="*50)
        print("🤖 AGENT RESPONSE")
        print("="*50)
        # Use your regex logic to clean up the output formatting
        for line in re.split(r'(?<=[.!?])\s+', output):
            line = line.strip()
            if line:
                print(f"  ➤  {line}")
        print("="*50 + "\n")

    except Exception as e:
        print(f"❌ Error: {e}")

/tmp/ipykernel_6997/265269883.py:68: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(


✅ Agent ready! Personality 'Gorq' is now active.
Start texting (type 'exit' or 'quit' to stop)
--------------------------------------------------
You: helo

🤖 AGENT RESPONSE
  ➤  Hello.
  ➤  Is there something I can help you with, or would you like to chat?

You: what can you do for me

🤖 AGENT RESPONSE
  ➤  I can help you with a variety of things.
  ➤  I can answer questions, provide information, and even help with tasks such as checking the weather or searching for something online.
  ➤  What's on your mind?
  ➤  Want to know the weather in a specific city or learn something new?

You: i want to buy a car red colour , budget is 10l

🤖 AGENT RESPONSE
  ➤  You're looking to buy a red car with a budget of 10 lakhs.
  ➤  There are many options available in the market.
  ➤  You can consider cars like the Maruti Suzuki Swift, Hyundai Grand i10, or the Ford Figo, which are all available in red and fall within your budget.
  ➤  Would you like me to search for more options or specific details